In [8]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, PreTrainedTokenizerFast, get_linear_schedule_with_warmup
from dataset import NMTDataset, nmt_collate_fn
from train import train_model
from models import Model1_GloVe, Model2_GloVeAttn, Model3_FrozenBERT, Model4_TunedBERT

In [9]:
def main():
    '''Select GPU (CUDA) if available; otherwise use the CPU for training'''
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"--- Device: {device} ---")

    '''Directory containing the train, validation, and test dataset CSV files'''
    DATA_DIR = "./nmtdataset"
    
    '''Load the trained BPE tokenizer used to tokenize target sentences into token IDs'''
    tgt_tokenizer = PreTrainedTokenizerFast.from_pretrained("tokenizer_target")
    '''Load the pretrained BERT tokenizer used to tokenize source sentences for the BERT encoder'''
    bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

    '''Load the saved GloVe vocabulary and embedding matrix'''
    glove_ckpt = torch.load("glove_vocab.pt")
    glove_word2idx = glove_ckpt["word2idx"]
    glove_matrix = glove_ckpt["matrix"]
    tgt_vocab_size = tgt_tokenizer.vocab_size

    '''Create the train, validation, and test datasets using GloVe tokenization for the source and BPE tokenization for the target'''
    glove_train = NMTDataset(f"{DATA_DIR}/Hindi_train.csv", tgt_tokenizer, src_tokenizer_mode="glove", glove_word2idx=glove_word2idx, is_train=True)
    glove_val   = NMTDataset(f"{DATA_DIR}/Hindi_dev.csv",   tgt_tokenizer, src_tokenizer_mode="glove", glove_word2idx=glove_word2idx)
    glove_test  = NMTDataset(f"{DATA_DIR}/Hindi_public_test.csv", tgt_tokenizer, src_tokenizer_mode="glove", glove_word2idx=glove_word2idx)

    '''Create DataLoaders that load, pad, and serve the dataset in mini-batches'''
    glove_train_loader = DataLoader(glove_train, batch_size=64, shuffle=True,  collate_fn=nmt_collate_fn)
    glove_val_loader   = DataLoader(glove_val,   batch_size=64, shuffle=False, collate_fn=nmt_collate_fn)
    glove_test_loader  = DataLoader(glove_test,  batch_size=1,  shuffle=False, collate_fn=nmt_collate_fn)

    '''Create the train, validation, and test datasets/DataLoaders using BERT tokenization for the source and BPE tokenization for the target'''
    bert_train = NMTDataset(f"{DATA_DIR}/Hindi_train.csv", tgt_tokenizer, src_tokenizer_mode="bert", bert_tokenizer=bert_tokenizer, is_train=True)
    bert_val   = NMTDataset(f"{DATA_DIR}/Hindi_dev.csv",   tgt_tokenizer, src_tokenizer_mode="bert", bert_tokenizer=bert_tokenizer)
    bert_test  = NMTDataset(f"{DATA_DIR}/Hindi_public_test.csv", tgt_tokenizer, src_tokenizer_mode="bert", bert_tokenizer=bert_tokenizer)

    '''Create DataLoaders that load, pad, and serve the dataset in mini-batches'''
    bert_train_loader = DataLoader(bert_train, batch_size=32, shuffle=True,  collate_fn=nmt_collate_fn)
    bert_val_loader   = DataLoader(bert_val,   batch_size=32, shuffle=False, collate_fn=nmt_collate_fn)
    bert_test_loader  = DataLoader(bert_test,  batch_size=1,  shuffle=False, collate_fn=nmt_collate_fn)

    '''Training configurations for each NMT model, specifying the model, dataset loaders, and teacher forcing strategy'''
    configs = [
        ("Model1_GloVe",     Model1_GloVe(glove_matrix, tgt_vocab_size),     glove_train_loader, glove_val_loader, glove_test_loader, "fixed"),
        ("Model2_GloVeAttn", Model2_GloVeAttn(glove_matrix, tgt_vocab_size), glove_train_loader, glove_val_loader, glove_test_loader, "fixed"),
        ("Model3_FrozenBERT", Model3_FrozenBERT(tgt_vocab_size),             bert_train_loader,  bert_val_loader,  bert_test_loader,  "fixed"),
        ("Model4_TunedBERT",  Model4_TunedBERT(tgt_vocab_size),              bert_train_loader,  bert_val_loader,  bert_test_loader,  "sigmoid"),
    ]

    histories = {}
    for name, model, tr_loader, vl_loader, _, tf_mode in configs:
        
        print(f"\n{'='*20} Training {name} {'='*20}")
        '''Number of complete passes through the entire training dataset'''
        num_epochs = 2
        
        if name == "Model4_TunedBERT":
            '''Train for fewer epochs since the pretrained BERT encoder is being fine-tuned'''
            num_epochs = 15
            '''
            Instead of using one learning rate, Adam is given two parameter groups - BERT and non-BERT parameters
            BERT is already well-trained. Large updates could destroy its pretrained knowledge.
            non-BERT layers are newly initialized, so they need to learn much faster.
            '''
            optimizer = optim.Adam([
                {'params': model.bert.parameters(), 'lr': 1e-5},
                {'params': [p for n, p in model.named_parameters() if not n.startswith('bert.')], 'lr': 5e-4}
            ])
            '''Total number of training batches (optimizer update steps) across all epochs'''
            total_steps = len(tr_loader) * num_epochs
            '''
            First 1000 steps → learning rate gradually increases (warmup).
            After that → learning rate gradually decreases linearly until training ends.
            '''
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=1000, num_training_steps=total_steps)
        elif name == "Model3_FrozenBERT":
            num_epochs = 20
            '''
            for param in self.bert.parameters():
                param.requires_grad = False
            i.e, BERT is frozen, the optimizer never updates BERT's weights
            '''
            optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
            '''Reduce the learning rate by half if the validation BLEU score does not improve for 3 consecutive epochs'''
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=3
            )
        else:
            '''This is the Adam optimizer, which updates the model parameters using the computed gradients during backpropagation.'''
            optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
            '''Reduce the learning rate by half if the validation BLEU score does not improve for 3 consecutive epochs'''
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=3
            )
        '''Finally, train the model and save the training history.'''
        histories[name] = train_model(
            model, tr_loader, vl_loader, optimizer, device,
            num_epochs=num_epochs, save_path=f"best_{name}.pth",
            scheduler=scheduler, tf_mode=tf_mode, fixed_tf_ratio=0.6,
            tgt_tokenizer=tgt_tokenizer
        )
        
    '''save the histories of each model training to anaylze graphs and scores'''
    torch.save(histories, "training_histories.pt")
    print("\nAll done. Histories saved to training_histories.pt")



In [ ]:
if __name__ == "__main__":
    main()

--- Device: cpu ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

C:\Users\karti\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\karti\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]


==================== Training Model1_GloVe ====================


Train:   1%|▊                                                                       | 10/909 [01:56<2:47:22, 11.17s/it]